In [2]:
%pip install ucimlrepo


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Dict
"""
    Load the Adult Census Income dataset from UCI repository.
    Reference:
        https://github.com/uci-ml-repo/ucimlrepo/blob/main/src/demo.ipynb
    """
# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# 1. Data Loading 
def load_adult_dataset() -> pd.DataFrame:
    
    adult = fetch_ucirepo(id=2)
    
    if adult.data is None:
        raise ValueError("Failed to fetch dataset from UCI repository.")
    
    # Combine features and targets
    df = pd.concat([adult.data.features, adult.data.targets], axis=1)
    
    return df
#Explortion of dataset

def explore_dataset(df: pd.DataFrame) -> Dict[str, any]:
    """
    Returns:
        dict: Summary statistics including column types, shape, etc.
    """
    # Identify column types
    cat_cols = [col for col in df.columns if df[col].dtype == 'object']
    num_cols = [col for col in df.columns if df[col].dtype != 'object']
    
    summary = {
        'shape': df.shape,
        'categorical_columns': cat_cols,
        'numerical_columns': num_cols,
        'total_samples': df.shape[0],
        'total_features': df.shape[1]
    }
    
#Display overview
    print("1. DATASET OVERVIEW")
    display(df.head(10))
    print(f"\nColumn Data Types:")
    print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
    print(f"Numerical columns ({len(num_cols)}): {num_cols}")
    print(f"\nDataset Summary:")
    print(f"Total samples: {summary['total_samples']:,}")
    print(f"Total features: {summary['total_features']}")
    
    return summary

# Load and explore data
df = load_adult_dataset()
dataset_summary = explore_dataset(df)


1. DATASET OVERVIEW


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
5,37,Private,284582,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,<=50K
6,49,Private,160187,9th,5,Married-spouse-absent,Other-service,Not-in-family,Black,Female,0,0,16,Jamaica,<=50K
7,52,Self-emp-not-inc,209642,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,45,United-States,>50K
8,31,Private,45781,Masters,14,Never-married,Prof-specialty,Not-in-family,White,Female,14084,0,50,United-States,>50K
9,42,Private,159449,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,5178,0,40,United-States,>50K



Column Data Types:
Categorical columns (9): ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']
Numerical columns (6): ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

Dataset Summary:
Total samples: 48,842
Total features: 15


In [ ]:
#2. Cleanining missing values on dataset
"""
Clean missing values in the dataset by replacing specified placeholders with NaN.
if found any missing values then drop those rows.
Args:
df: Input DataFrame
    missing_value: Placeholder for missing values (default is '?')
    Returns:
        pd.DataFrame: Cleaned DataFrame with missing values handled
        Reference:
        https://www.geeksforgeeks.org/data-analysis/data-cleansing-introduction/
    """
def analyze_missing_values(df: pd.DataFrame, missing_value: str = '?') -> pd.DataFrame:
    
    print("2. MISSING VALUE ANALYSIS")

    missing_info = []
    
#Check object columns for missing value
    object_cols = df.select_dtypes(include=['object']).columns
    
    for col in object_cols:
        missing_count = (df[col] == missing_value).sum()
        if missing_count > 0:
            missing_pct = (missing_count / len(df)) * 100
            missing_info.append({
                'column': col,
                'missing_count': missing_count,
                'missing_percentage': missing_pct
            })
            print(f"{col}: {missing_count:,} ({missing_pct:.2f}%)")
    
#Check for NaN values
    nan_counts = df.isnull().sum()
    if nan_counts.sum() > 0:
        print("\nNaN values found:")
        print(nan_counts[nan_counts > 0])
    
    if not missing_info:
        print("No missing values found.")
        
    return pd.DataFrame(missing_info)

#Analyze missing values
missing_summary = analyze_missing_values(df)

2. MISSING VALUE ANALYSIS
workclass: 1,836 (3.76%)
occupation: 1,843 (3.77%)
native-country: 583 (1.19%)

NaN values found:
workclass         963
occupation        966
native-country    274
dtype: int64


In [64]:
#3. Data cleaning functions
"""
Remove specified columns "we don't need them" for x reason of analysis   
Args: columns_to_remove: List of column names to remove      
Returns: pd.DataFrame: DataFrame with columns removed      
Note:
 Columns removed FOR NOW are:
 - 'fnlwgt': Census sampling weight (not useful for analysis)
 - 'education-num': Duplicate of 'education' column (numerical encoding)
    """

def remove_columns(df: pd.DataFrame, columns_to_remove: List[str]) -> pd.DataFrame:
   
    return df.drop(columns=columns_to_remove, errors='ignore')


def handle_missing_values(df: pd.DataFrame, 
                         missing_marker: str = '?',
                         strategy: str = 'mode') -> pd.DataFrame:
    
    """
    Handle missing values in the dataset.
    Args:
     df: Input DataFrame
     missing_marker: String representing missing values (default: '?')
    strategy: Method to handle missing values
    - 'mode': Fill with most frequent value (default)
    - 'drop': Remove rows with missing values
    Reference:
        https://medium.com/@ugamakelechi501/how-to-prepare-and-clean-datasets-for-machine-learning-6ce2b7192d80
    """
    # Replace missing marker with NaN
    df = df.replace(missing_marker, np.nan)
    
    if strategy == 'drop':
        # More aggressive: removes all rows with any missing values "REVIEW THIS"
        df = df.dropna()
        
    elif strategy == 'mode':
        # Fill categorical columns with mode (most frequent value)
        for col in df.select_dtypes(include='object').columns:
            mode_values = df[col].mode()
            
            if len(mode_values) > 0:
                df[col] = df[col].fillna(mode_values[0])
            else:
                # If no mode exists, fill with 'Unknown'
                df[col] = df[col].fillna('Unknown')
    
    return df

def remove_duplicates(df: pd.DataFrame) -> Tuple[pd.DataFrame, int]:
    """
    Remove duplicate rows   
    Returns:
        tuple: (cleaned DataFrame, number of duplicates removed)
    """
    duplicates_count = df.duplicated().sum()
    df_cleaned = df.drop_duplicates()
    
    return df_cleaned, duplicates_count


def generate_cleaning_report(rows_before: int, 
                            rows_after: int,
                            duplicates_removed: int,
                            columns_removed: List[str]) -> None:
    """
    Generate and print a summary report of the cleaning process.
    
    Args:
        rows_before: Number of rows before cleaning
        rows_after: Number of rows after cleaning
        duplicates_removed: Number of duplicate rows removed
        columns_removed: List of column names that were removed
    """
    rows_removed = rows_before - rows_after
    data_retention = (rows_after / rows_before) * 100
    
    print(f"4. CLEANING SUMMARY")

    print(f"\nColumns removed: {columns_removed}")
    print(f"Duplicate rows removed: {duplicates_removed:,}")
    print(f"Total rows removed: {rows_removed:,}")
    print(f"Data retention: {data_retention:.2f}%")
    print(f"Total samples after data cleaning: {rows_after:,} ")



In [65]:
# 4. Execute Cleaning Pipeline
""" Execute complete data cleaning pipeline.
    Pipeline Steps:
        1. Remove irrelevant columns
        2. Handle missing values
        3. Remove duplicate rows
        4. Generate cleaning report
    """
def clean_adult_dataset(df: pd.DataFrame, 
                       columns_to_remove: List[str] = ['fnlwgt', 'education-num'],
                       missing_strategy: str = 'mode') -> pd.DataFrame:
   
   
    print("3. DATA CLEANING PIPELINE\n")
    
    # Store initial state
    rows_before = len(df)

    # Step 1: Remove columns
    df_cleaned = remove_columns(df, columns_to_remove)
    print(f"Removed {len(columns_to_remove)} columns")
    
    # Step 2: Handle missing values
    df_cleaned = handle_missing_values(df_cleaned, strategy=missing_strategy)
    print(f"Applied '{missing_strategy}' strategy for missing values")
    
    # Step 3: Remove duplicates
    df_cleaned, duplicates_removed = remove_duplicates(df_cleaned)
    print(f"Removed {duplicates_removed:,} duplicate rows\n")
    
    # Generate report
    rows_after = len(df_cleaned)
    generate_cleaning_report(rows_before, rows_after, duplicates_removed, columns_to_remove)
    
    return df_cleaned

# %%
# Execute cleaning pipeline
df_cleaned = clean_adult_dataset(df, missing_strategy='mode')

3. DATA CLEANING PIPELINE

Removed 2 columns
Applied 'mode' strategy for missing values
Removed 4,655 duplicate rows

4. CLEANING SUMMARY

Columns removed: ['fnlwgt', 'education-num']
Duplicate rows removed: 4,655
Total rows removed: 4,655
Data retention: 90.47%
Total samples after data cleaning: 44,187 



### Pandas Methods Used:

- df.drop() - Remove columns
- df.replace() - Replace values
- df.dropna() - Remove missing values
- df.str.strip() - Remove whitespace
- df.isnull() - Check for NaN
- df.value_counts() - Count unique values
- df.describe() - Summary statistics
- df.duplicated() - Check for duplicate rows
- df.drop_duplicates() - Remove duplicate rows

https://medium.com/@ugamakelechi501/how-to-prepare-and-clean-datasets-for-machine-learning-6ce2b7192d80


In [ ]:
"""""
# Checking the data before cleaning
# Check for missing values (marked as '?')
print("\nMissing values check per feature:")

# Convert to list to ensure it's iterable
object_columns = list(df.select_dtypes(include=['object']).columns)

for col in object_columns:
    missing_count = (df[col] == '?').sum()
    if missing_count > 0:
        missing_pct = round((missing_count / df.shape[0]) * 100, 2)
        print(f"{col}: {missing_count:,} ({missing_pct}%)")
        """


Missing values check per feature:
workclass: 1,836 (3.76%)
occupation: 1,843 (3.77%)
native-country: 583 (1.19%)


In [ ]:
# If one of our targets is income, let's standardize it
# Standardize income column (remove whitespace and trailing periods)
#df_cleaned['income'] = df_cleaned['income'].str.strip().str.rstrip('.')

#print("Income distribution:")
#print(df_cleaned['income'].value_counts())
#print(f" Income categories: {df_cleaned['income'].nunique()}")

#index by sex   
#df.set_index('sex', inplace = True)
#df.head()


Columns that were removed: ['education-num', 'fnlwgt']

Reasons:
  - education-num: Duplicate of 'education' column (numerical encoding)
  - fnlwgt: Census sampling weigh


Future cleaning for a specific target such as age, gender,education...

Identification: For numerical columns like 'age', 'fnlwgt', 'capital-gain', 'capital-loss', and 'hours-per-week', we will need to identify extreme values that might be data entry errors or genuine but unusual observations 
. For example, an 'age' of 0 or an 'hours-per-week' of 1000 would be considered an outlier.


2. Standardizing Categorical Data
Categorical variables often contain inconsistencies that need standardization.

Inconsistent Entries: Check for variations in spelling, capitalization, or different representations of the same category (e.g., "United-States" vs. "United States" or "HS-grad" vs. "HS Grad"). The provided data consistently uses "United-States" and "HS-grad"